# 🚀 Optimized LSTM Training with Validation

Training script for Optimized LSTM model using CuPy (GPU).  
Uses batched operations for ~10-50x speedup.  
Includes train/validation split and visualization for overfitting analysis.

## 1. Setup & Imports

In [1]:
import numpy as np
import pandas as pd
import os
import sys
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# Add src to path for model import
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))
from model.lstm_cupy_optimized import LSTMModelGPUOptimized, GPU_AVAILABLE

print(f"GPU Available: {GPU_AVAILABLE}")

✅ CuPy detected - Using GPU acceleration
GPU Available: True


## 2. Configuration

In [2]:
# Paths
DATA_DIR = os.path.join(os.getcwd(), '..', 'data')
PREPROCESSED_DIR = os.path.join(DATA_DIR, 'preprocessed')
OUTPUT_FILE = os.path.join(DATA_DIR, 'predictions_gpu_optimized.csv')
PLOT_FILE = os.path.join(DATA_DIR, 'training_history.png')

# Hyperparameters (adjust as needed)
HIDDEN_SIZE = 64
EPOCHS = 100
LEARNING_RATE = 0.001
BATCH_SIZE = 128
VAL_SPLIT = 0.2
RANDOM_STATE = 42

print("Configuration:")
print(f"  Hidden Size:   {HIDDEN_SIZE}")
print(f"  Epochs:        {EPOCHS}")
print(f"  Learning Rate: {LEARNING_RATE}")
print(f"  Batch Size:    {BATCH_SIZE}")
print(f"  Val Split:     {VAL_SPLIT}")

Configuration:
  Hidden Size:   64
  Epochs:        100
  Learning Rate: 0.001
  Batch Size:    128
  Val Split:     0.2


## 3. Load Data

In [3]:
print("Loading preprocessed data...")
X = np.load(os.path.join(PREPROCESSED_DIR, 'X_sequences.npy'))
y = np.load(os.path.join(PREPROCESSED_DIR, 'y_sequences.npy'))

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Class distribution: 0={np.sum(y==0)} | 1={np.sum(y==1)}")
print(f"Class ratio: {np.mean(y):.2%} positive")

Loading preprocessed data...
X shape: (39254, 5, 6)
y shape: (39254,)
Class distribution: 0=27021 | 1=12233
Class ratio: 31.16% positive


## 4. Train/Validation Split

In [4]:
print(f"Splitting data ({int((1-VAL_SPLIT)*100)}/{int(VAL_SPLIT*100)})...")

X_train, X_val, y_train, y_val = train_test_split(
    X, y, 
    test_size=VAL_SPLIT, 
    random_state=RANDOM_STATE,
    stratify=y  # Maintain class balance
)

print(f"Training:   {len(X_train)} samples")
print(f"Validation: {len(X_val)} samples")
print(f"Train class distribution: 0={np.sum(y_train==0)} | 1={np.sum(y_train==1)}")
print(f"Val class distribution:   0={np.sum(y_val==0)} | 1={np.sum(y_val==1)}")

Splitting data (80/20)...
Training:   31403 samples
Validation: 7851 samples
Train class distribution: 0=21617 | 1=9786
Val class distribution:   0=5404 | 1=2447


## 5. Initialize Model

In [5]:
input_size = X.shape[2]
seq_len = X.shape[1]

print(f"Input size:      {input_size}")
print(f"Sequence length: {seq_len}")
print(f"Hidden size:     {HIDDEN_SIZE}")

model = LSTMModelGPUOptimized(input_size=input_size, hidden_size=HIDDEN_SIZE)
print("\n✅ Model initialized!")

Input size:      6
Sequence length: 5
Hidden size:     64

✅ Model initialized!


## 6. Training

In [ ]:
print("Starting training...")
print(f"Epochs: {EPOCHS}, Batch size: {BATCH_SIZE}, Learning rate: {LEARNING_RATE}")
print()

history = model.train(
    X_train, y_train,
    X_val, y_val,
    epochs=EPOCHS, 
    batch_size=BATCH_SIZE,
    lr=LEARNING_RATE,
    print_every=10
)

Starting training...
Epochs: 100, Batch size: 128, Learning rate: 0.001

Training: 31403 samples, 246 batches/epoch, batch_size=128
Validation: 7851 samples
----------------------------------------------------------------------
Epoch   10/100 | Train Loss: 0.5944 | Val Loss: 0.5974 | Val Acc: 0.6910 [UNDERFITTING]
Epoch   20/100 | Train Loss: 0.5898 | Val Loss: 0.5932 | Val Acc: 0.6957 [UNDERFITTING]
Epoch   30/100 | Train Loss: 0.5855 | Val Loss: 0.5936 | Val Acc: 0.6948 [UNDERFITTING]


## 7. Training Visualization

In [ ]:
epochs_range = range(1, len(history['train_loss']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Loss comparison
ax1 = axes[0]
ax1.plot(epochs_range, history['train_loss'], 'b-', label='Training Loss', linewidth=2)
if history['val_loss']:
    ax1.plot(epochs_range, history['val_loss'], 'r-', label='Validation Loss', linewidth=2)
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Loss (BCE)', fontsize=12)
ax1.set_title('Training vs Validation Loss', fontsize=14)
ax1.legend(loc='upper right', fontsize=10)
ax1.grid(True, alpha=0.3)

# Add annotation for overfitting detection
if history['val_loss']:
    final_train = history['train_loss'][-1]
    final_val = history['val_loss'][-1]
    gap = final_val - final_train
    
    if gap > 0.1:
        ax1.annotate('⚠️ Overfitting Detected', 
                    xy=(len(epochs_range), final_val), 
                    xytext=(len(epochs_range)*0.7, max(history['val_loss'])*0.9),
                    fontsize=10, color='red',
                    arrowprops=dict(arrowstyle='->', color='red'))

# Plot 2: Accuracy over epochs
ax2 = axes[1]
if history['val_acc']:
    ax2.plot(epochs_range, history['val_acc'], 'g-', label='Validation Accuracy', linewidth=2)
if history['train_acc']:
    ax2.plot(epochs_range, history['train_acc'], 'b--', label='Training Accuracy', linewidth=2, alpha=0.7)
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Accuracy', fontsize=12)
ax2.set_title('Accuracy over Epochs', fontsize=14)
ax2.legend(loc='lower right', fontsize=10)
ax2.grid(True, alpha=0.3)
ax2.set_ylim([0, 1])
ax2.axhline(y=0.5, color='gray', linestyle='--', linewidth=1, label='Random Baseline')

plt.tight_layout()
plt.savefig(PLOT_FILE, dpi=150, bbox_inches='tight')
plt.show()

print(f"\n📊 Plot saved to: {PLOT_FILE}")

## 8. Final Evaluation

In [ ]:
# Evaluate on validation set
val_loss, val_acc = model.evaluate(X_val, y_val, batch_size=BATCH_SIZE)
print(f"Validation Loss:     {val_loss:.4f}")
print(f"Validation Accuracy: {val_acc:.4f}")

In [ ]:
# Generate predictions on validation set
probabilities = model.predict(X_val, batch_size=BATCH_SIZE)
predictions = (probabilities >= 0.5).astype(int)

print("Classification Report:")
print(classification_report(y_val, predictions, target_names=['Class 0', 'Class 1']))

In [ ]:
# Confusion Matrix Visualization
cm = confusion_matrix(y_val, predictions)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
ax.figure.colorbar(im, ax=ax)

ax.set(xticks=[0, 1], yticks=[0, 1],
       xticklabels=['Predicted 0', 'Predicted 1'],
       yticklabels=['Actual 0', 'Actual 1'],
       title='Confusion Matrix',
       ylabel='True label',
       xlabel='Predicted label')

# Add text annotations
thresh = cm.max() / 2.
for i in range(2):
    for j in range(2):
        ax.text(j, i, format(cm[i, j], 'd'),
                ha="center", va="center",
                color="white" if cm[i, j] > thresh else "black",
                fontsize=20)

plt.tight_layout()
plt.show()

print(f"\nConfusion Matrix:")
print(f"  TN={cm[0,0]}, FP={cm[0,1]}")
print(f"  FN={cm[1,0]}, TP={cm[1,1]}")

## 9. Diagnosis: Overfitting vs Underfitting

In [ ]:
final_train_loss = history['train_loss'][-1]
final_val_loss = history['val_loss'][-1]
gap = final_val_loss - final_train_loss

print("=" * 60)
print("DIAGNOSIS")
print("=" * 60)
print(f"Final Training Loss:   {final_train_loss:.4f}")
print(f"Final Validation Loss: {final_val_loss:.4f}")
print(f"Gap (Val - Train):     {gap:.4f}")

if val_acc < 0.6 and gap < 0.1:
    print("\n⚠️ DIAGNOSIS: UNDERFITTING")
    print("   - Both train and validation performance are poor")
    print("   - Model is too simple to capture patterns")
    print("   - Suggestions:")
    print(f"     * Increase hidden_size (currently: {HIDDEN_SIZE})")
    print("     * Train for more epochs")
    print("     * Check data quality and preprocessing")
    print("     * Add more features if available")
elif gap > 0.1:
    print("\n⚠️ DIAGNOSIS: OVERFITTING")
    print("   - Training loss is much lower than validation loss")
    print("   - Model memorizes training data instead of generalizing")
    print("   - Suggestions:")
    print("     * Add dropout regularization")
    print("     * Reduce hidden_size")
    print("     * Use early stopping")
    print("     * Collect more training data")
else:
    print("\n✅ Model appears to be well-fitted")
    print("   - If accuracy is still low, consider:")
    print("     * Feature engineering")
    print("     * Hyperparameter tuning")
    print("     * Trying different architectures")

## 10. Save Predictions

In [ ]:
# Save predictions to CSV
results_df = pd.DataFrame({
    'actual': y_val,
    'predicted': predictions,
    'probability': probabilities
})

results_df.to_csv(OUTPUT_FILE, index=False)
print(f"✅ Predictions saved to: {OUTPUT_FILE}")

# Preview
results_df.head(10)

## 11. Summary Statistics

In [ ]:
print("=" * 60)
print("TRAINING SUMMARY")
print("=" * 60)
print(f"Model:           LSTM (hidden_size={HIDDEN_SIZE})")
print(f"GPU:             {GPU_AVAILABLE}")
print(f"Training samples: {len(X_train)}")
print(f"Validation samples: {len(X_val)}")
print(f"Epochs:          {EPOCHS}")
print(f"Batch size:      {BATCH_SIZE}")
print(f"Learning rate:   {LEARNING_RATE}")
print()
print(f"Final Train Loss: {final_train_loss:.4f}")
print(f"Final Val Loss:   {final_val_loss:.4f}")
print(f"Final Val Acc:    {val_acc:.4f}")
print("=" * 60)